# Service 1 — Étape 2 : évaluation du modèle de base (AVANT fine-tuning)

On pose les **109 questions de test** à Qwen-2.5-7B-Instruct **tel quel** (aucun entraînement).
Les scores obtenus sont la **référence** : à l'étape 4, on refera exactement la même chose avec le modèle fine-tuné.

| Type de question | Nombre | Mesure |
|---|---|---|
| Service 3 (JSON) | 40 | JSON valide / complet, résultat, cas limite et fiabilité justes |
| MedMCQA (QCM) | 50 | Bonne lettre |
| MedQuAD (questions patients) | 19 | ROUGE-L (ressemblance avec la réponse de référence) |

**Avant de lancer :** *Exécution → Modifier le type d'exécution → GPU T4*, puis *Exécution → Tout exécuter*.
Durée : environ 30 à 45 min (le modèle écrit ses réponses une par une). Les réponses sont sauvegardées au fur et à mesure sur le Drive : si Colab se déconnecte, relancer reprend où on en était.

## 1. Drive, code et chemins

In [1]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/cardioagent-ops"
REPO_URL = "https://github.com/yasssmine2003/CardioAgent-Ops.git"
REPO_DIR = "/content/CardioAgent-Ops"
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}/llm_finetuning

EVAL_DIR = f"{DRIVE}/llm/eval"
os.makedirs(EVAL_DIR, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE}/mlruns"
print("Test présent :", os.path.exists("data/final/test.jsonl"))

Mounted at /content/drive
/content/CardioAgent-Ops/llm_finetuning
Test présent : True


## 2. Installation (Unsloth, MLflow, ROUGE)
Les avertissements rouges de pip sur des conflits de versions sont habituels sur Colab.

In [2]:
!pip install -q unsloth "mlflow>=2.17,<3" rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
bigframes 2.51.0 requires pyarrow>=23.0.1, but you have pyarrow 19.0.1 which is incompatible.


## 3. Chargement du modèle de base en 4 bits
Version déjà quantifiée en 4 bits par Unsloth : environ 5,5 Go à télécharger, tient dans le GPU T4 (16 Go).

In [3]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # [A] Qwen-2.5-7B-Instruct, quantifié 4 bits
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)                  # mode « réponse » (plus rapide)
print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1119 13:18:42.641000 586 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1119 13:18:42.679000 586 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Mémoire GPU utilisée : 5.6 Go


## 4. Génération des réponses (sauvegarde au fur et à mesure)

In [4]:
import json
from src.evaluation import generate_answers

STAGE = "baseline"                                   # étape 4 : "finetuned"
pred_path = f"{EVAL_DIR}/{STAGE}_predictions.jsonl"

with open("data/final/test.jsonl", encoding="utf-8") as f:
    test = [json.loads(line) for line in f]

done = []
if os.path.exists(pred_path):                        # reprise après déconnexion
    with open(pred_path, encoding="utf-8") as f:
        done = [json.loads(line) for line in f]
print(f"{len(done)} réponses déjà faites, {len(test) - len(done)} à faire")

for start in range(len(done), len(test), 10):        # par paquets de 10, sauvegardés tout de suite
    batch = generate_answers(model, tokenizer, test[start:start + 10], log_every=100)
    with open(pred_path, "a", encoding="utf-8") as f:
        for r in batch:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{start + len(batch)}/{len(test)} sauvegardées")

0 réponses déjà faites, 109 à faire


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

10/109 sauvegardées


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

20/109 sauvegardées


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

... (13 outputs left out)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

100/109 sauvegardées


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

109/109 sauvegardées


## 5. Scores

In [5]:
import pandas as pd
from src.evaluation import score_all

with open(pred_path, encoding="utf-8") as f:
    results = [json.loads(line) for line in f]
results, summary = score_all(results)
summary["memoire_gpu_max_go"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)

with open(f"{EVAL_DIR}/{STAGE}_scores.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
pd.Series(summary, name=STAGE).to_frame()

,baseline
service3_json_valide,1.0000
service3_json_complet,0.0000
service3_resultat_juste,0.0000
service3_cas_limite_juste,0.0000
service3_fiabilite_juste,0.0000
service3_n,40.0000
medmcqa_bonne_lettre,0.5600
medmcqa_n,50.0000
medquad_rougeL,0.1715
medquad_n,19.0000


## 6. Quelques réponses à lire (une de chaque type)

In [6]:
for family in ["service3", "medmcqa", "medquad"]:
    r = next(x for x in results if x["family"] == family)
    print(f"===== {family} =====\nQUESTION :\n{r['question'][:500]}\n\nRÉPONSE DU MODÈLE :\n{r['prediction'][:700]}"
          f"\n\nRÉFÉRENCE :\n{r['reference'][:400]}\n\nSCORES : {r['scores']}\n")

===== service3 =====
QUESTION :
Analyse ce résultat du moteur audio (enregistrement phonocardiographique) et réponds en JSON.

Résultat du moteur audio :
{"prediction": "pathologique", "probability": 0.9148, "confidence": 0.9148, "segment_probabilities": [0.8342, 0.8873, 0.9596, 0.8969, 0.919, 0.992], "n_segments": 6, "duration_s": 34.71, "padded": false, "model": {"threshold": 0.7307}}

RÉPONSE DU MODÈLE :
{
  "analyse": {
    "prediction": "pathologique",
    "probability": 0.9148,
    "confidence": 0.9148,
    "segments": [
      {"probability": 0.8342},
      {"probability": 0.8873},
      {"probability": 0.9596},
      {"probability": 0.8969},
      {"probability": 0.919},
      {"probability": 0.992}
    ],
    "duration": 34.71,
    "model_threshold": 0.7307
  }
}

RÉFÉRENCE :
{
  "resultat": "pathologique",
  "fiabilite": "moyenne",
  "cas_limite": false,
  "interpretation": "Probabilité « pathologique » de 0,915, au-dessus du seuil de décision (0,731), avec un écart modéré de 

## 7. Enregistrement dans MLflow

In [7]:
import mlflow
from src.evaluation import MAX_NEW_TOKENS

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("cardioagent-llm")
with mlflow.start_run(run_name=f"eval_{STAGE}"):
    mlflow.set_tags({"stage": STAGE, "model": MODEL_NAME, "family": "evaluation"})
    mlflow.log_params({"n_test": len(results), "decoding": "greedy",
                       **{f"max_new_tokens_{k}": v for k, v in MAX_NEW_TOKENS.items()}})
    mlflow.log_metrics(summary)
    mlflow.log_artifact(pred_path)
    mlflow.log_artifact(f"{EVAL_DIR}/{STAGE}_scores.json")
print("Run MLflow enregistré :", f"eval_{STAGE}")

2026/11/19 13:28:38 INFO mlflow.tracking.fluent: Experiment with name 'cardioagent-llm' does not exist. Creating a new experiment.


Run MLflow enregistré : eval_baseline
